# Dependencies

In [220]:
from datasets import load_dataset
from dotenv import load_dotenv
from typing import Callable, Any
from pathlib import Path
import torch as T
import string
import os

# Data
Importing the data

In [221]:
load_dotenv("./.env.secrets")

hf_token = os.getenv("HF_TOKEN")

FORMAT = "torch"

ds = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-v1",
    token=hf_token if hf_token else False,
).with_format(FORMAT)

x_test, x_train, x_val = (ds[key] for key in ds.keys())

display(x_train["text"][:10])

['',
 ' = Valkyria Chronicles III = \n',
 '',
 ' Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " <unk> Raven " . \n',
 " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving

## Data pre-processing
Preprocessing the data allows us to collect as much useful information as possible, these following operations are performed:

- Remove punctuations: Since our aim is to analyze the semantic meaning of individual words and not sentences. Including punctuation, which carries meaning in a sentences, would be misleading since we would lead to situations where two different words (for example "apple." and "apple") have slightly different meanings when they, when taken out of a sentence context, should have practically the same meaning. This is also computationally inficient if you consider that our task is strictly confined to analyzing the semantic difference and meaning of words.

- Force lower: We do this for exactly the same reasons as why we remove punctuation.

You could make the case that the words "Apple", "apple", "apple." actually do have differenet semanting meanings and technically you would be correct, but as stated above, we want to focus the model to learn the difference between "apple" and "pear" rather than using its parameters to differentiate between "Apple" and "apple".

In [222]:
def pre_process_data(data: T.utils.data.Dataset):
    # We clean the words in the dataset a bit, this is to ensure that all words present
    # in the vocabulary are in a predictable format
    to_remove = string.punctuation + "“”‘’"
    table = str.maketrans("", "", to_remove)  # Create a translation table
    big_blob = " ".join(data["text"])
    # Translate the the blob using the table and convert to lower
    cleaned_blob = big_blob.translate(table).lower()
    return sorted(list(set(cleaned_blob.split())))

# One-hot-encoding model
Assigns each word in a vocabulary a $n$ dimensional vector with a single element set to one (uniquely), the words are sorted alphabetically.

In [223]:
class OneHotEncoder:
    def __init__(self, vocabulary: list[str]):
        self.vocabulary = vocabulary
        self.dim = len(vocabulary)
        self.word_indexes = {word: i for i, word in enumerate(vocabulary)}

    def __call__(self, word: str):
        vec = T.zeros(self.dim)
        idx = self.word_indexes.get(word)
        if idx:
            vec[idx] = 1
        return vec

In [224]:
FN = "one_hot_encoder.pt"
file_path = Path(FN)
if file_path.exists():
   encoder = T.load(file_path, weights_only=False)
else:
    vocabulary = pre_process_data(x_train)
    encoder = OneHotEncoder(vocabulary)
    T.save(encoder, file_path)

# Defining the modeling tools
In a Object Oriented way

In [225]:
class Derivable():

    def __init__(self):

        self.dydi: T.Tensor = None
        """
        Derivative (D) of the output y (D) w.r.t to the Input (DI) of the object.
        """

        self.dydw: T.Tensor = None
        """
        Derivative (D) of the output y (Y) w.r.t to the Weights (DW) of the object.
        """

In [226]:
class Trainable(Derivable):

    def update_weights(self, dnydi: T.Tensor) -> None:
        """
        Update the weights of the object using the derivative of the output of the
        next layer w.r.t its input which is the current layer output.
        """
        raise NotImplementedError(
            f"{self.__name__} has not implemented update_weights"
        )

## Softmax
Softmax is a function that turns a vector of numbers $\mathbf{z} \in \mathbf{R}^n$ in to a vector of proabilities $\mathbf{\hat{y}}$ s.t $\sum_i \hat{y}_i = 1$. The softmax function is defined as:

$$\mathbf{\hat{y}}_i = \frac{e^{z_i}}{\sum_{j=1}^{n} e^{z_j}}$$

The gradients are calculated automatically during the forward pass and are set follows:

$$
    \begin{align*}
        & \delta_{ij} = \begin{cases} 1 & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases} \\
        & \frac{\partial \hat{y}_i}{\partial z_j} = \hat{y}_i(\delta_{ij} - \hat{y}_j) \\
        & \text{diag}(\mathbf{\hat{y}}) = \begin{bmatrix} \hat{y}_1 & 0 & \dots \\ 0 & \hat{y}_2 & \dots \\ \vdots & \vdots & \ddots \end{bmatrix}
        & \mathbf{\hat{y}}\mathbf{\hat{y}}^T = \begin{bmatrix} \hat{y}_1^2 & \hat{y}_1\hat{y}_2 & \dots \\ \hat{y}_2\hat{y}_1 & \hat{y}_2^2 & \dots \\ \vdots & \vdots & \ddots \end{bmatrix}\\
        & \frac{\partial{\mathbf{\hat{y}}}}{\partial{\mathbf{z}}} = \text{diag}(\mathbf{\hat{y}}) - \mathbf{\hat{y}}\mathbf{\hat{y}}^T
    \end{align*}
$$

The single element version $\frac{\partial \hat{y}_i}{\partial z_j}$ of the derivative is included for clarity as it makes it easier to understand the full vector ($\mathbf{\hat{y}}$). 

In [227]:
class Softmax(Derivable, Callable[[T.Tensor], T.Tensor]):
    def __call__(self, z: T.Tensor) -> T.Tensor:
        # Exponentiate all elements of z, z are the logits, or the "raw" input.
        z_exp = T.exp(z)
        # Squeeze away the last element of y, this makes no difference mathematically.
        y = (z_exp / T.sum(z_exp, dim=0)).squeeze()
        # Store the partial derivative dy/dz.
        self.dydi = T.diag(y) - T.outer(y.T,y)
        return y


sm = Softmax()
test_values = [
    T.tensor([1.0, 0.0, 0.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 0.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 1.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 1.0, 1.0]).reshape(-1, 1),
]

for val in test_values:
    out = sm(val)
    dydi = sm.dydi
    print(f"Output: {out},\nPartial derivative: {dydi}")

Output: tensor([0.4754, 0.1749, 0.1749, 0.1749]),
Partial derivative: tensor([[ 0.2494, -0.0831, -0.0831, -0.0831],
        [-0.0831,  0.1443, -0.0306, -0.0306],
        [-0.0831, -0.0306,  0.1443, -0.0306],
        [-0.0831, -0.0306, -0.0306,  0.1443]])
Output: tensor([0.3655, 0.3655, 0.1345, 0.1345]),
Partial derivative: tensor([[ 0.2319, -0.1336, -0.0492, -0.0492],
        [-0.1336,  0.2319, -0.0492, -0.0492],
        [-0.0492, -0.0492,  0.1164, -0.0181],
        [-0.0492, -0.0492, -0.0181,  0.1164]])
Output: tensor([0.2969, 0.2969, 0.2969, 0.1092]),
Partial derivative: tensor([[ 0.2088, -0.0882, -0.0882, -0.0324],
        [-0.0882,  0.2088, -0.0882, -0.0324],
        [-0.0882, -0.0882,  0.2088, -0.0324],
        [-0.0324, -0.0324, -0.0324,  0.0973]])
Output: tensor([0.2500, 0.2500, 0.2500, 0.2500]),
Partial derivative: tensor([[ 0.1875, -0.0625, -0.0625, -0.0625],
        [-0.0625,  0.1875, -0.0625, -0.0625],
        [-0.0625, -0.0625,  0.1875, -0.0625],
        [-0.0625, -0.0625, 

## Linear layer
The object `LinearLayer2D` performs the following operation on an input vector $\mathbf{v}$:

$$
    \begin{align*}
        & \mathbf{z} = \mathbf{v}^T \mathbf{W} + \mathbf{b} \\
        & \mathbf{\hat{y}} = \sigma{(\mathbf{z})}
    \end{align*}
$$

Where $\mathbf{b}$ is the bias term, $\mathbf{W}$ is the weights, and $\mathbf{\sigma}$ is the activation function, $\mathbf{z}$ is just an intermediary step used for convenience.

We have, however, modified this operation slightly so that we can remove the separate bias term and include it directly in an expanded weight matrix $\mathbf{W_b}$. This is achieved by adding a row filled with ones to $\mathbf{W}$. So the operation actually becomes:

$$
    \begin{align*}
        & \mathbf{z} = \mathbf{v}^T \mathbf{W_b} \\
        & \mathbf{\hat{y}} = \sigma{(\mathbf{z})}
    \end{align*}
$$

The gradients are calculated automatically during the forward pass and are set as follows:

$$
    \begin{align*}
        & \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{W_b}} = \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}} \cdot \mathbf{v} \\
        & \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{v}} = \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}} \cdot \mathbf{W_b}
    \end{align*}
$$

In [ ]:
class LinearLayer2D(Trainable):
    def __init__(
        self,
        dim_in: int,
        dim_out: int,
        activation: Derivable,
    ):
        self.shape = (dim_in,dim_out)
        # Add bias dim
        weight_dims = (dim_in + 1, dim_out)
        # Initialize weights
        self.weights = T.randn(
            size=weight_dims,
        )
        self.activation = activation

    def __call__(self, vec: T.Tensor) -> T.Tensor:
        # add 1 to the last dim and fill that with 1s to create a single
        # weight matrix
        ones = T.ones(1,)
        vec_plus_bias = T.cat((vec, ones), dim=0)
        # Get y hat
        y = self.activation((vec_plus_bias.T @ self.weights))
        # Store the derivatives on the forward pass
        #TODO fix this derivative error
        self.dydi = self.activation.dydi @ self.weights
        self.dydw = self.activation.dydi @ vec_plus_bias
        return y

    def update_weights(self, dnydi: T.Tensor) -> None:
        if self.dydw:
            self.weights -= dnydi @ self.dydw 
        else:
            raise RuntimeError("Perform a forward pass before trying to ")


test_values = [
    T.tensor([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], dtype=T.float32)
]

ll = LinearLayer2D(10, 3, Softmax())

for val in test_values:
    out = sm(val)
    dydi = sm.dydi
    print(f"Output: {out},\nPartial derivative: {dydi}")

Output: tensor([0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
        0.1000]),
Partial derivative: tensor([[ 0.0900, -0.0100, -0.0100, -0.0100, -0.0100, -0.0100, -0.0100, -0.0100,
         -0.0100, -0.0100],
        [-0.0100,  0.0900, -0.0100, -0.0100, -0.0100, -0.0100, -0.0100, -0.0100,
         -0.0100, -0.0100],
        [-0.0100, -0.0100,  0.0900, -0.0100, -0.0100, -0.0100, -0.0100, -0.0100,
         -0.0100, -0.0100],
        [-0.0100, -0.0100, -0.0100,  0.0900, -0.0100, -0.0100, -0.0100, -0.0100,
         -0.0100, -0.0100],
        [-0.0100, -0.0100, -0.0100, -0.0100,  0.0900, -0.0100, -0.0100, -0.0100,
         -0.0100, -0.0100],
        [-0.0100, -0.0100, -0.0100, -0.0100, -0.0100,  0.0900, -0.0100, -0.0100,
         -0.0100, -0.0100],
        [-0.0100, -0.0100, -0.0100, -0.0100, -0.0100, -0.0100,  0.0900, -0.0100,
         -0.0100, -0.0100],
        [-0.0100, -0.0100, -0.0100, -0.0100, -0.0100, -0.0100, -0.0100,  0.0900,
         -0.0100, -0.0100],
  

## Model

In [229]:
class Model:
    def __init__(self, layers: list[Trainable | Derivable | object]):
        self.layers = layers

    def __call__(self, input: T.Tensor):
        curr = input
        for l in self.layers:
            curr = l(curr)
        return curr

## Training harness
An object that trains a Model object using stochastic gradient descent (SGD)
 
```py
for epoch in nr_epochs:
    for b in get_batch(training_data):
        model.forward(b)
        dnydi = tensor([1])
        for l in model.layers.reverse():
            if l is Trainable:
                l.update_weights(dnydi)
            dnydi = dnydi @ l.dydi
```

# Creating the Word2Vec model

In [230]:
ONE_HOT_DIM = encoder.dim
OUT_DIM = 10
word2vec = Model([
    OneHotEncoder(encoder.vocabulary), 
    LinearLayer2D(
        dim_in=ONE_HOT_DIM,
        dim_out=OUT_DIM,
        activation=Softmax()
    )
])

display(word2vec("banana"))

tensor([0.1068, 0.0121, 0.0328, 0.0484, 0.0041, 0.6363, 0.0497, 0.0737, 0.0029,
        0.0332])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (10x10 and 226922x10)

# Training, Validation, Testing

In [ ]:
#TODO implement regularization so we can select a lambda param during validation
#TODO try different output sizes on the linear layer

# Visualizing results

## PCA
...

## UMAP
...

In [ ]:
#TODO visualize results using UMAP and PCA